In [2]:
### IMPORTS ###
import sys
sys.path.append('../')

import autolib
import fom
import Optimisation.opt as opt
import pathlib
#from pathlib import PosixPath
#user_home_path = PosixPath('~/')
#user_home_path_full = user_home_path.expanduser()


#final_speed = 20.
num_cores = 200
maxtime = 1435
#runID = "Fasympmonochrome_fixgaussian20_50GW_nG30"
runID = "Fasymp2_fixgaussian20_50GW" # _nG30"

student = "Grating_Half"  # Change this to your name or preferred folder name

# common_path = user_home_path_full / "Library/CloudStorage/OneDrive-TheUniversityofSydney(Students)/Doppler Damping - Jadon Lin/Documentation/Data/relativistic-lightsail-dynamics/Optimisation/Jadon's results"
# custom_folder_path = f"Fasymp/final_speed{int(final_speed)}/maxtime{int(maxtime)}/{runID}"
# custom_folder_path = f"Fasymp/mono/maxtime{int(maxtime)}/{runID}"

filepath = "Z:\\Github\\relativistic-lightsail-dynamics\\Optimisation\\run_parallel_extract.ipynb"
filepath = "combine_extract.ipynb"
current_dir = pathlib.Path(filepath).resolve(strict=True).parent
fname_preamble = current_dir / "Data" / student

# fname_preamble = common_path / custom_folder_path

# import pathlib
# fname_preamble = pathlib.Path("./Data")

pkl_fname = fname_preamble / f'{runID}_FOM_optimisation_maxtime{maxtime}'
txt_fname = fname_preamble / f'{runID}_FOM_optimisation_maxtime{maxtime}_curated.txt'

maxima_and_maximisers_sorted, opt_gratings_sorted, _ = opt.extract_opt(pkl_fname, num_processes=num_cores, output_opt_idx=0)

Total function evaluations: 731622
Average function evaluations per core: 3658


In [3]:
def print_grating_params(grating):
    print(f"grating_pitch   = {grating.params[0]}")
    print(f"grating_depth   = {grating.params[1]}")
    print(f"box1_width      = {grating.params[2]}")
    print(f"box2_width      = {grating.params[3]}")
    print(f"box_centre_dist = {grating.params[4]}")
    print(f"box1_eps        = {grating.params[5]}")
    print(f"box2_eps        = {grating.params[6]}")
    print(f"gaussian_width  = {grating.params[7]}")
    print(f"substrate_depth = {grating.params[8]}")
    print(f"substrate_eps   = {grating.params[9]}")

In [4]:
import numpy as np
from scipy.optimize import minimize
from parameters import D1_ND

def get_final(l_min, l_max):
    doppler = l_min / l_max

    def test(v):
        v = v/100
        return np.abs(doppler - D1_ND(v))

    sol = minimize(test, 5)
    final_speed = sol.x[0]
    return final_speed

In [28]:
import numpy as np
import os
os.environ["OMP_NUM_THREADS"] = "1" 
os.environ["OPENBLAS_NUM_THREADS"] = "1" 
os.environ["MKL_NUM_THREADS"] = "1" 
os.environ["VECLIB_MAXIMUM_THREADS"] = "1" 
os.environ["NUMEXPR_NUM_THREADS"] = "1" 

import parameters
from parameters import D1_ND

from twobox import TwoBox

I0, L, m, c = parameters.Parameters()
optimum_number = 17

final_speed = get_final(1., 1.05)

wavelength_range = [1,1/D1_ND([final_speed/100,0])]
#wavelength_range = [0.99999,1.0000004]
#bandwidth = wavelength_range[1] - wavelength_range[0]
#relative_bandwidth = bandwidth / wavelength_range[0]
#wavelength_range = [0.99999 - 10.*relative_bandwidth, 1.0000004 + 10.*relative_bandwidth]
print(wavelength_range)
_, _, opt_grating = opt.extract_opt(pkl_fname, num_processes=num_cores, output_opt_idx=optimum_number)
optimisation_RCWA_engine = opt_grating.RCWA_engine
print(optimisation_RCWA_engine)

# Test optimum FOM convergence by rebuilding the grating with higher fidelity
match optimisation_RCWA_engine:
    case "GRCWA":  # GRCWA opt params are saved as autograd ArrayBoxes, have to get values manually
        opt_params = [p._value for p in opt_grating.all_params]
    case _:
        opt_params = opt_grating.all_params
print(opt_params)
print(opt_grating.params)
grating_opt17 = TwoBox(*opt_params, wavelength=1., angle=0., Nx=opt_grating.Nx, nG=opt_grating.nG, Qabs=np.inf, RCWA_engine=opt_grating.RCWA_engine, torcwa_edge_sharpness=opt_grating.torcwa_edge_sharpness)
grating_opt17converged = TwoBox(*opt_params, wavelength=1., angle=0., Nx=1000, nG=100, Qabs=np.inf, RCWA_engine=opt_grating.RCWA_engine, torcwa_edge_sharpness=opt_grating.torcwa_edge_sharpness)    

print(fom.multifom(grating_opt17, monofom=fom.monofom_asymp, final_speed=final_speed, goal=0.1, return_grad=False))
# print(fom.multifom(grating_converged, monofom=fom.monofom_asymp, final_speed=final_speed, goal=0.1, return_grad=False))
print(f"grating_pitch   = {grating_opt17.params[0]}")
print(f"grating_depth   = {grating_opt17.params[1]}")
print(f"box1_width      = {grating_opt17.params[2]}")
print(f"box2_width      = {grating_opt17.params[3]}")
print(f"box_centre_dist = {grating_opt17.params[4]}")
print(f"box1_eps        = {grating_opt17.params[5]}")
print(f"box2_eps        = {grating_opt17.params[6]}")
print(f"gaussian_width  = {grating_opt17.params[7]}")
print(f"substrate_depth = {grating_opt17.params[8]}")
print(f"substrate_eps   = {grating_opt17.params[9]}")
print(f"\nFoM recorded:     {maxima_and_maximisers_sorted[optimum_number][0]}")

[1, np.float64(1.049999999971945)]
Total function evaluations: 731622
Average function evaluations per core: 3658
TORCWA
[tensor(1.6442, dtype=torch.float64), tensor(0.4207, dtype=torch.float64), tensor(0.4393, dtype=torch.float64), tensor(0.0699, dtype=torch.float64), tensor(0.7073, dtype=torch.float64), tensor(6.2141, dtype=torch.float64), tensor(10.3459, dtype=torch.float64), tensor(20., dtype=torch.float64), tensor(0.7074, dtype=torch.float64), tensor(5.0511, dtype=torch.float64)]
[tensor(1.6442, dtype=torch.float64), tensor(0.4207, dtype=torch.float64), tensor(0.4393, dtype=torch.float64), tensor(0.0699, dtype=torch.float64), tensor(0.7073, dtype=torch.float64), tensor(6.2141, dtype=torch.float64), tensor(10.3459, dtype=torch.float64), tensor(0.7074, dtype=torch.float64), tensor(5.0511, dtype=torch.float64)]
0.00012346665548080174
grating_pitch   = 1.644158094108683
grating_depth   = 0.42065441113361174
box1_width      = 0.43931427220999403
box2_width      = 0.06986396941011432
bo

In [29]:
scaled = np.sqrt(1.05) # Scale to new wavelength
scaled_opt_params = opt_params.copy()

param_ls = [0, 1, 2, 3, 4, 7, 8]
for p in param_ls:
    scaled_opt_params[p] = scaled * scaled_opt_params[p]
    
scaled_grating_opt17 = TwoBox(*scaled_opt_params, wavelength=1., angle=0., 
                        Nx=opt_grating.Nx, nG=opt_grating.nG, Qabs=np.inf,
                        RCWA_engine=opt_grating.RCWA_engine, 
                        torcwa_edge_sharpness=opt_grating.torcwa_edge_sharpness)

final_speed = get_final(1., 1.05)

actual_fom = fom.multifom(scaled_grating_opt17, monofom=fom.monofom_asymp,
                            final_speed=final_speed, goal=0.1, return_grad=False)

print_grating_params(scaled_grating_opt17)
print(f"\nFoM at scaled wavelength: {actual_fom}")

grating_pitch   = 1.6847607041785646
grating_depth   = 0.4310425040369847
box1_width      = 0.45016317181191823
box2_width      = 0.0715892654859949
box_centre_dist = 0.7247638860623848
box1_eps        = 6.214070093349006
box2_eps        = 10.345909666634435
gaussian_width  = 20.4939015319192
substrate_depth = 0.7249060383871563
substrate_eps   = 5.0511030085969155

FoM at scaled wavelength: -187.4432412868258


In [12]:
stiffnesses_opt17 = fom.force_coeff(grating_opt17,I0,m,c,grad_method="finite",out="mat",normalise=False) # to get jacobian/matrix

J_original_opt17 = grating_opt17.npa.concatenate((grating_opt17.npa.array([[0,0,1,0],[0,0,0,1]]), stiffnesses_opt17))

In [13]:
stiffnesses_scaled_opt17 = fom.force_coeff(scaled_grating_opt17,I0,m,c,grad_method="finite",out="mat",normalise=False) # to get jacobian/matrix

J_scaled_opt17 = scaled_grating_opt17.npa.concatenate((scaled_grating_opt17.npa.array([[0,0,1,0],[0,0,0,1]]), stiffnesses_scaled_opt17))

In [15]:
eigvals_opt17 = grating_opt17.npa.eigvals((J_scaled_opt17 + J_original_opt17)/2)
eigReal_opt17 = grating_opt17.npa.real(eigvals_opt17)
eigImag_opt17 = grating_opt17.npa.imag(eigvals_opt17)
eigvals_opt17

tensor([-0.0004+165.8330j, -0.0004-165.8330j, -0.0002+29.1797j,
        -0.0002-29.1797j], dtype=torch.complex128,
       grad_fn=<LinalgEigBackward0>)

In [24]:
fom_combined =  grating_opt17.npa.min(-eigReal_opt17)
print('FoM of combined (scaled), opt17 =',fom_combined.detach().numpy())

FoM of combined (scaled), opt17 = 0.00017301544803437565


Making a loop for different optimisations:

In [45]:
I0, L, m, c = parameters.Parameters()
optimum_number = np.arange(0,50)

final_speed = get_final(1., 1.05)

#FoM_array = []
avg_FoM_array = []

wavelength_range = [1,1/D1_ND([final_speed/100,0])]
#wavelength_range = [0.99999,1.0000004]
#bandwidth = wavelength_range[1] - wavelength_range[0]
#relative_bandwidth = bandwidth / wavelength_range[0]
#wavelength_range = [0.99999 - 10.*relative_bandwidth, 1.0000004 + 10.*relative_bandwidth]

for i in range(0,len(optimum_number)):
    #print(wavelength_range)
    _, _, opt_grating = opt.extract_opt(pkl_fname, num_processes=num_cores, output_opt_idx=i)
    optimisation_RCWA_engine = opt_grating.RCWA_engine
    print(optimisation_RCWA_engine)

    # Test optimum FOM convergence by rebuilding the grating with higher fidelity
    match optimisation_RCWA_engine:
        case "GRCWA":  # GRCWA opt params are saved as autograd ArrayBoxes, have to get values manually
            opt_params = [p._value for p in opt_grating.all_params]
        case _:
            opt_params = opt_grating.all_params

    grating = TwoBox(*opt_params, wavelength=1., angle=0., Nx=opt_grating.Nx, nG=opt_grating.nG, Qabs=np.inf, RCWA_engine=opt_grating.RCWA_engine, torcwa_edge_sharpness=opt_grating.torcwa_edge_sharpness)
    grating_converged = TwoBox(*opt_params, wavelength=1., angle=0., Nx=1000, nG=100, Qabs=np.inf, RCWA_engine=opt_grating.RCWA_engine, torcwa_edge_sharpness=opt_grating.torcwa_edge_sharpness)    

    scaled = np.sqrt(1.05) # Scale to new wavelength
    scaled_opt_params = opt_params.copy()

    param_ls = [0, 1, 2, 3, 4, 7, 8]
    for p in param_ls:
        scaled_opt_params[p] = scaled * scaled_opt_params[p]
        
    scaled_grating = TwoBox(*scaled_opt_params, wavelength=1., angle=0., 
                            Nx=opt_grating.Nx, nG=opt_grating.nG, Qabs=np.inf,
                            RCWA_engine=opt_grating.RCWA_engine, 
                            torcwa_edge_sharpness=opt_grating.torcwa_edge_sharpness)

    final_speed = get_final(1., 1.05)

    num_plot_points = 200

    wavelengths = np.linspace(*wavelength_range, num_plot_points)
    init_wavelength = 1.
    #eigvals = grating.npa.zeros((4,num_plot_points), dtype=np.complex128)
    #eigReal = grating.npa.zeros((4,num_plot_points), dtype=np.complex128)
    #FoMs = grating.npa.zeros((4,num_plot_points), dtype=np.complex128)
    eigs = []
    eigs_Real = []
    FoMs = []

    for idx, lam in enumerate(wavelengths):
        # Calculate eigs for each order
        grating.wavelength = grating.npa.array(lam)
        scaled_grating.wavelength = scaled_grating.npa.array(lam)
        
        stiffnesses = fom.force_coeff(grating,I0,m,c,grad_method="finite",out="mat",normalise=False) # to get jacobian/matrix

        J_original = grating.npa.concatenate((grating.npa.array([[0,0,1,0],[0,0,0,1]]), stiffnesses))

        stiffnesses_scaled = fom.force_coeff(scaled_grating,I0,m,c,grad_method="finite",out="mat",normalise=False) # to get jacobian/matrix

        J_scaled = scaled_grating.npa.concatenate((scaled_grating.npa.array([[0,0,1,0],[0,0,0,1]]), stiffnesses_scaled))

        #eigvals[:,idx] = grating.npa.eigvals((J_scaled + J_original)/2)
        #eigReal[:,idx] = grating.npa.real(eigvals)

        #FoMs[:,idx] =  grating.npa.min(-eigReal)

        eigvals = grating.npa.eigvals((J_scaled + J_original)/2)
        eigReal = grating.npa.real(eigvals)

        FoM = grating.npa.min(-eigReal)
        eigs_Real.append(eigReal)
        FoMs.append(FoM.detach().numpy())

        
    grating.wavelength = init_wavelength
    scaled_grating.wavelength = init_wavelength
    eigvals

    '''
    stiffnesses = fom.force_coeff(grating,I0,m,c,grad_method="finite",out="mat",normalise=False) # to get jacobian/matrix

    J_original = grating.npa.concatenate((grating.npa.array([[0,0,1,0],[0,0,0,1]]), stiffnesses))

    stiffnesses_scaled = fom.force_coeff(scaled_grating,I0,m,c,grad_method="finite",out="mat",normalise=False) # to get jacobian/matrix

    J_scaled = scaled_grating.npa.concatenate((scaled_grating.npa.array([[0,0,1,0],[0,0,0,1]]), stiffnesses_scaled))

    eigvals = grating.npa.eigvals((J_scaled + J_original)/2)
    eigReal = grating.npa.real(eigvals)
    '''

    #fom_combined =  grating.npa.min(-eigReal)
    average_foms = np.mean(FoMs)
    avg_FoM_array.append(average_foms)

    #print('FoM of combined (scaled), opt',i,' =',fom_combined.detach().numpy())
    print('average FoM of combined (scaled), opt',i,' =',average_foms)

    #FoM_array.append(fom_combined.detach().numpy())

Total function evaluations: 731622
Average function evaluations per core: 3658
TORCWA
average FoM of combined (scaled), opt 0  = -12.166493353368827
Total function evaluations: 731622
Average function evaluations per core: 3658
TORCWA
average FoM of combined (scaled), opt 1  = -25.54336586164879
Total function evaluations: 731622
Average function evaluations per core: 3658
TORCWA
average FoM of combined (scaled), opt 2  = -15.870467366251592
Total function evaluations: 731622
Average function evaluations per core: 3658
TORCWA
average FoM of combined (scaled), opt 3  = -10.693927229031473
Total function evaluations: 731622
Average function evaluations per core: 3658
TORCWA
average FoM of combined (scaled), opt 4  = -7.751530262464654
Total function evaluations: 731622
Average function evaluations per core: 3658
TORCWA
average FoM of combined (scaled), opt 5  = -8.984554673342439
Total function evaluations: 731622
Average function evaluations per core: 3658
TORCWA
average FoM of combined

In [51]:
np.max(avg_FoM_array)
len(FoMs)

200

In [52]:
import pandas as pd
df_FoMs = pd.DataFrame(avg_FoM_array)

df_FoMs

,0
0,-12.166493
1,-25.543366
2,-15.870467
3,-10.693927
4,-7.751530
5,-8.984555
6,-66.243090
7,-47.595253
8,-7.788221
9,-53.499776
